# 02 - Exploratory analysis

This notebook gives a first look at the shared monthly sample. The four sleeves are long-only index returns. Small Cap is a market segment, not an SMB return series.

## Setup and sample coverage

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from robust_dm_factor_allocation import (
    load_config,
    load_monthly_returns,
    performance_metrics,
    plot_wealth,
)

In [ ]:
working_directory = Path.cwd().resolve()
repository_root = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(
    config.data_file,
    factor_columns=config.factor_columns,
    market_column=config.benchmark,
    missing=config.missing_policy,
)

tables_directory = repository_root / "results" / "tables"
figures_directory = repository_root / "results" / "figures"
tables_directory.mkdir(parents=True, exist_ok=True)
figures_directory.mkdir(parents=True, exist_ok=True)

In [ ]:
coverage = pd.DataFrame(
    {
        "observations": monthly_returns.notna().sum(),
        "first_month": monthly_returns.apply(pd.Series.first_valid_index),
        "last_month": monthly_returns.apply(pd.Series.last_valid_index),
    }
)
coverage

## Return, risk and correlation

Annualized return is the geometric rate, $\left[\prod_{t=1}^{T}(1+r_t)\right]^{12/T}-1$. Annualized volatility is the monthly sample standard deviation multiplied by $\sqrt{12}$. Maximum drawdown is the largest fall from a previous peak in the compounded wealth index. Sharpe and Sortino ratios use monthly arithmetic excess returns and are also annualized by $\sqrt{12}$.

The matrix below uses Pearson correlation between returns from the same month. It measures linear co-movement, not causality, and the relationship can change over time.

In [ ]:
periods_per_year = config.periods_per_year
return_summary = monthly_returns.apply(
    performance_metrics,
    periods_per_year=periods_per_year,
).T
monthly_correlation = monthly_returns.corr(method="pearson")

return_summary.to_csv(tables_directory / "eda_return_summary.csv")
monthly_correlation.to_csv(tables_directory / "eda_monthly_correlation.csv")
return_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_wealth(monthly_returns, ax=ax, log_scale=True)
ax.set_title("Common-sample wealth indices")
fig.tight_layout()
fig.savefig(figures_directory / "eda_wealth_indices.png", dpi=160)
plt.close(fig)

initial_wealth = pd.DataFrame(
    1.0,
    index=[monthly_returns.index[0] - pd.offsets.MonthEnd(1)],
    columns=monthly_returns.columns,
)
wealth = pd.concat([initial_wealth, (1 + monthly_returns).cumprod()])
drawdowns = wealth.div(wealth.cummax()).sub(1)
fig, ax = plt.subplots(figsize=(11, 6))
drawdowns.plot(ax=ax)
ax.set_title("Common-sample drawdowns")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(figures_directory / "eda_drawdowns.png", dpi=160)
plt.close(fig)

In [ ]:
labels = monthly_correlation.columns.tolist()
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(monthly_correlation, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(np.arange(len(labels)), labels=labels, rotation=45, ha="right")
ax.set_yticks(np.arange(len(labels)), labels=labels)
ax.set_title("Pearson correlation of monthly returns")
fig.colorbar(image, ax=ax, label="Correlation")
fig.tight_layout()
fig.savefig(figures_directory / "eda_monthly_correlation.png", dpi=160)
plt.close(fig)

monthly_correlation

## Notes

I use these tables to compare return, risk, drawdown and correlation before building the portfolios. Full-sample correlation can hide stronger co-movement in stressed markets.

The metadata marks histories that start before the official index launch. Notebook 04 also reports post-launch results.